# 1. 패키지 설치

In [1]:
# 패키지 설치
%pip install -U python-dotenv langchain langchain-openai langchain-community langchain-text-splitters docx2txt langchain-chroma langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Knowledge Base 구성을 위한 데이터 생성

- markdown으로 변환한 docx(`tax_with_markdown.docx`)를 chunking 후 Chroma에 저장

In [3]:
# docx 문서 Load & Split
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax_with_markdown.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
# 환경변수 Load + Embedding 설정
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [ ]:
# Vector DB(Chroma)
from langchain_chroma import Chroma

# 데이터 처음 저장 (최초 1회)
# database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory='./chroma_markdown')

# 이미 저장된 데이터 사용 (2회차부터는 위를 주석 처리하고 아래 사용)
database = Chroma(collection_name='chroma-tax', persist_directory='./chroma_markdown', embedding_function=embedding)

# 3. 답변 생성을 위한 Retrieval

In [6]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'

In [7]:
# create_retrieval_chain용 retriever 생성 (k=10: 검색 문서 10개)
retriever = database.as_retriever(search_kwargs={'k': 10})
retriever.invoke(query)

[Document(id='4d51dbe1-9020-4348-a2d4-e1567e468d3f', metadata={'source': './tax_with_markdown.docx'}, page_content='나. 그 밖의 배당소득에 대해서는 100분의 14\n\n3. 원천징수대상 사업소득에 대해서는 100분의 3. 다만, 외국인 직업운동가가 한국표준산업분류에 따른 스포츠 클럽 운영업 중 프로스포츠구단과의 계약(계약기간이 3년 이하인 경우로 한정한다)에 따라 용역을 제공하고 받는 소득에 대해서는 100분의 20으로 한다.\n\n4. 근로소득에 대해서는 기본세율. 다만, 일용근로자의 근로소득에 대해서는 100분의 6으로 한다.\n\n5. 공적연금소득에 대해서는 기본세율\n\n5의2.제20조의3제1항제2호나목 및 다목에 따른 연금계좌 납입액이나 운용실적에 따라 증가된 금액을 연금수령한 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 각 목의 요건을 동시에 충족하는 때에는 낮은 세율을 적용한다.\n\n가. 연금소득자의 나이에 따른 다음의 세율\n\n\n\n나. 삭제<2014. 12. 23.>\n\n다. 사망할 때까지 연금수령하는 대통령령으로 정하는 종신계약에 따라 받는 연금소득에 대해서는 100분의 4\n\n5의3. 제20조의3제1항제2호가목에 따라 퇴직소득을 연금수령하는 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 연금 실제 수령연차 및 연금외수령 원천징수세율의 구체적인 내용은 대통령령으로 정한다.\n\n가. 연금 실제 수령연차가 10년 이하인 경우: 연금외수령 원천징수세율의 100분의 70\n\n나. 연금 실제 수령연차가 10년을 초과하는 경우: 연금외수령 원천징수세율의 100분의 60\n\n6. 기타소득에 대해서는 다음에 규정하는 세율. 다만, 제8호를 적용받는 경우는 제외한다.\n\n가. 제14조제3항제8호라목 및 마목에 해당하는 소득금액이 3억원을 초과하는 경우 그 초과하는 분에 대해서는 100분의 30\n\n나. 제21조제1

# 4. Augmentation을 위한 Prompt 활용

- `create_retrieval_chain` 전용 프롬프트(`langchain-ai/retrieval-qa-chat`, 변수 input+context)를 Hub에서 가져옴

In [8]:
# 프롬프트: create_retrieval_chain용 retrieval-qa-chat (System + chat_history + Human, 변수 input·context)
# langchain.hub 제거 / langchain_classic.hub.pull deprecated → langsmith로 pull
from langsmith import Client

client = Client()
retrieval_qa_chat_prompt = client.pull_prompt('langchain-ai/retrieval-qa-chat', dangerously_pull_public_prompt=True)

In [9]:
# LangChain 기반 LLM 설정
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

# 5. 답변 생성을 위한 RAG 체인 구성

- `create_stuff_documents_chain`(문서 결합) + `create_retrieval_chain`(검색→생성)으로 RAG 체인 구성
    - 실제 답변 호출은 6단계에서 키워드 보정(`dictionary_chain`)과 연계해 수행 (01~03에서 다룬 단순 답변 생성은 생략 — 04는 키워드 사전 보정이 핵심)
    - `RetrievalQA`(3.2)의 대체. v1에서 `langchain_classic`에 위치하나 deprecated 아님

In [11]:
# v1: create_stuff_documents_chain·create_retrieval_chain 은 langchain_classic 으로 이동 (deprecated 아님)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [12]:
# (보정 전) '직장인' 질문을 그대로 사용 → 세율표 매칭이 약해 부정확할 수 있음
# (출력 dict의 키: input / context / answer)
ai_message = retrieval_chain.invoke({'input': query})
ai_message

{'input': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'context': [Document(id='4d51dbe1-9020-4348-a2d4-e1567e468d3f', metadata={'source': './tax_with_markdown.docx'}, page_content='나. 그 밖의 배당소득에 대해서는 100분의 14\n\n3. 원천징수대상 사업소득에 대해서는 100분의 3. 다만, 외국인 직업운동가가 한국표준산업분류에 따른 스포츠 클럽 운영업 중 프로스포츠구단과의 계약(계약기간이 3년 이하인 경우로 한정한다)에 따라 용역을 제공하고 받는 소득에 대해서는 100분의 20으로 한다.\n\n4. 근로소득에 대해서는 기본세율. 다만, 일용근로자의 근로소득에 대해서는 100분의 6으로 한다.\n\n5. 공적연금소득에 대해서는 기본세율\n\n5의2.제20조의3제1항제2호나목 및 다목에 따른 연금계좌 납입액이나 운용실적에 따라 증가된 금액을 연금수령한 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 각 목의 요건을 동시에 충족하는 때에는 낮은 세율을 적용한다.\n\n가. 연금소득자의 나이에 따른 다음의 세율\n\n\n\n나. 삭제<2014. 12. 23.>\n\n다. 사망할 때까지 연금수령하는 대통령령으로 정하는 종신계약에 따라 받는 연금소득에 대해서는 100분의 4\n\n5의3. 제20조의3제1항제2호가목에 따라 퇴직소득을 연금수령하는 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 연금 실제 수령연차 및 연금외수령 원천징수세율의 구체적인 내용은 대통령령으로 정한다.\n\n가. 연금 실제 수령연차가 10년 이하인 경우: 연금외수령 원천징수세율의 100분의 70\n\n나. 연금 실제 수령연차가 10년을 초과하는 경우: 연금외수령 원천징수세율의 100분의 60\n\n6. 기타소득에 대해서는 다음에 규정하는 세율. 다만, 제8호를 적용받는 경우는 제외한다.\n\n가. 제14조제3항제8호라목 및 마목에 해당하는 소득금액

# 6. Retrieval을 위한 keyword 사전 활용

- Knowledge Base에서 쓰는 용어로 사용자 질문을 보정(예: 직장인 → 거주자)
- LCEL로 `dictionary_chain`(질문 보정) → `qa_chain`(RAG) 연계

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ['사람을 나타내는 표현 -> 거주자']

# 사전을 참고해 질문을 보정하는 프롬프트 (변수명을 dictionary_prompt로 두어 위 retrieval_qa_chat_prompt와 구분)
dictionary_prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전: {dictionary}

    질문: {{question}}
""")

dictionary_chain = dictionary_prompt | llm | StrOutputParser()
# 질문 보정 → retrieval_chain. create_retrieval_chain 입력키가 'input'이라 {"input": ...}로 매핑
tax_chain = {'input': dictionary_chain} | retrieval_chain

In [14]:
# 사전 적용 결과 확인: '직장인' → '거주자'로 보정되는지
new_question = dictionary_chain.invoke({'question': query})
new_question

'연봉 5천만원인 거주자의 소득세는 얼마인가요?'

In [15]:
# (보정 후) 질문 보정 + RAG 한 번에 → 더 정확한 답변 기대
ai_response = tax_chain.invoke({'question': query})
ai_response

{'input': '연봉 5천만원인 거주자의 소득세는 얼마인가요?',
 'context': [Document(id='5fa36073-7cc2-48ed-94c6-7002c4fa50fe', metadata={'source': './tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 40퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (